# Sleep data inventory

This notebook inspects the privacy-safe FIT and JSON-derived tables. It does not save personal outputs to the repository.

In [ ]:
from garmin_daylio_analysis.sleep import load_local_sleep_analysis

sleep = load_local_sleep_analysis()
nights, segments, signals = sleep.nights, sleep.segments, sleep.signals

In [ ]:
nights[["sleep_date", "fit_coverage", "summary_coverage"]].value_counts(dropna=False), signals.notna().mean().sort_values()

In [ ]:
# This cell can also be run on its own after restarting the kernel.
if not {"nights", "segments", "signals"}.issubset(globals()):
    from garmin_daylio_analysis.sleep import load_local_sleep_analysis

    sleep = load_local_sleep_analysis()
    nights, segments, signals = sleep.nights, sleep.segments, sleep.signals

if segments.empty:
    print("No FIT sleep-stage segments were included in this Garmin export. Sleep summaries are available below.")
else:
    display(segments.groupby("raw_stage").duration_minutes.describe())

In [ ]:
# Show a stage hypnogram when available; otherwise show the sleep-summary duration trend.
if not {"nights", "segments", "signals"}.issubset(globals()):
    from garmin_daylio_analysis.sleep import load_local_sleep_analysis

    sleep = load_local_sleep_analysis()
    nights, segments, signals = sleep.nights, sleep.segments, sleep.signals

import matplotlib.pyplot as plt
if segments.empty:
    plt.plot(nights.sleep_date, nights.summary_total_seconds / 3600, marker="o", markersize=3)
    plt.ylabel("Sleep-summary duration (hours)")
    plt.xlabel("Sleep date")
    plt.title("Garmin sleep-summary duration")
else:
    chosen_date = nights.sleep_date.max()
    plot = segments[segments.sleep_date == chosen_date].copy()
    stage_order = {"awake": 4, "rem": 3, "light": 2, "deep": 1, "unmeasurable": 0}
    plt.step(plot.start_local, plot.raw_stage.map(stage_order), where="post")
    plt.yticks(list(stage_order.values()), list(stage_order))
    plt.title(f"Garmin hypnogram: {chosen_date}")